In [1]:
import pandas as pd
from pathlib import Path
datadir = Path("data")
resolutions = "resolutions_flat.parquet"

Smart. Here's the plan:

## Training Set Alignment Plan

**Goal:** Extract (HTR_span, canonical_name, role, metadata) pairs from 1626–1630 resolutions for GysBERT fine-tuning.

### Phase 1: Load & Filter
1. Load resolutions_flat.parquet
2. Filter to year 1626–1630
3. Check structure:
   - Is there a `delegate_names` column (list/JSON)?
   - Is there a `text` column (HTR'd content)?
   - What keys link them (date, resolution_id)?

### Phase 2: Canonicalize & Extract Summaries
1. For each resolution, build a set of canonical delegate names from the summary
2. Optional: extract role info if available (attended as what?)
3. Store as `(date, resolution_id, canonical_names, roles_if_any)`

### Phase 3: Find Matches in HTR
1. For each resolution's HTR text, search for delegate name mentions
2. Use fuzzy matching (rapidfuzz token_set_ratio) to catch OCR variants
3. Extract spans: character offsets + matched text
4. Score confidence (exact match > fuzzy match)

### Phase 4: Build Training Pairs
1. For each matched span, create tuple:
   ```
   (htr_text, canonical_name, resolution_id, date, confidence)
   ```
2. Deduplicate (same name variant appearing multiple times)
3. Filter low-confidence matches or manual review threshold

### Phase 5: Output
- Save as parquet/JSON-L
- Statistics: how many pairs, coverage, variant patterns
- Manual spot-check (show some samples for validation)

**Questions before we code:**
- Do summaries have structured delegate lists, or parsed from narrative?
- Are roles recorded (e.g., "ambassador", "councillor")?
- How messy is the HTR — should we set a fuzzy threshold (e.g., only >70% token_set_ratio)?



In [2]:
resolutions_df = pd.read_parquet(resolutions)
resolutions_df.head()

,id,type,date,year,weekday,paragraph_texts,resolutions_text
0,session-3788-num-1-resolution-1,resolution,1733-01-02,1733,vrijdag,"[""ONtfangen een Missive van den Resident Spina...","ONtfangen een Missive van den Resident Spina, ..."
1,session-3788-num-1-resolution-10,resolution,1733-01-02,1733,vrijdag,"[""IS ter Vergaderinge gelesen de Requeste van ...",IS ter Vergaderinge gelesen de Requeste van de...
2,session-3788-num-1-resolution-11,resolution,1733-01-02,1733,vrijdag,"[""17 Ynde ter Vergaderinge getoont en geexhibe...",17 Ynde ter Vergaderinge getoont en geexhibeer...
3,session-3788-num-1-resolution-12,resolution,1733-01-02,1733,vrijdag,"[""17 Ynde ter Vergaderinge getoont ende geëxhi...",17 Ynde ter Vergaderinge getoont ende geëxhibe...
4,session-3788-num-1-resolution-13,resolution,1733-01-02,1733,vrijdag,"[""OP de Requeste van de gesamentlijcke Straatm...",OP de Requeste van de gesamentlijcke Straatmaa...


In [3]:
resolutions_df['year'].value_counts().sort_index()

year
1577    1661
1578     957
1579    1188
1580     753
1581     197
        ... 
1794    4082
1795    6614
1796    1192
1924       1
1982       1
Name: count, Length: 222, dtype: int64

In [4]:
# Inspect resolutions_flat schema
print("resolutions_flat columns:", resolutions_df.columns.tolist())
print("Shape:", resolutions_df.shape)
print("\nDtypes:", resolutions_df.dtypes)
print("\nFirst row sample:")
for col in ['date', 'volgnr', 'resolution_index', 'paragraphs_texts']:
    if col in resolutions_df.columns:
        sample = resolutions_df[col].iloc[0]
        if isinstance(sample, str):
            print(f"  {col}: {sample[:100]}...")
        else:
            print(f"  {col}: {sample}")


resolutions_flat columns: ['id', 'type', 'date', 'year', 'weekday', 'paragraph_texts', 'resolutions_text']
Shape: (692156, 7)

Dtypes: id                    str
type                  str
date                  str
year                int64
weekday               str
paragraph_texts       str
resolutions_text      str
dtype: object

First row sample:
  date: 1733-01-02...


In [5]:
import json
from pathlib import Path
from rapidfuzz.distance import Levenshtein
from rapidfuzz.fuzz import token_set_ratio

# Load summary resolutions (1626-1630, self-contained)
summary_path = datadir / "enriched_resolutions_1626_1630_complete.json"
with open(summary_path) as f:
    summaries = json.load(f)
print(f"Loaded {len(summaries)} enriched resolutions")

# Load metadata from local data dir (all 1626-1630 data)
deputies_info_path = datadir / "deputies_info.json"
with open(deputies_info_path) as f:
    deputies_info = json.load(f)
print(f"Loaded {len(deputies_info)} deputy records")

persoon_functie_path = datadir / "persoon_functie.json"
with open(persoon_functie_path) as f:
    persoon_functie = json.load(f)
print(f"Loaded {len(persoon_functie)} person-function mappings")

# Quick inspection
print("\nSample summary resolution:")
print(f"  Keys: {summaries[0].keys()}")
print(f"  Date: {summaries[0]['date']}")
print(f"  Deputy IDs: {summaries[0].get('deputy_ids', [])[:3]}")

print("\nSample deputy info:")
if isinstance(deputies_info, dict):
    sample_deputy_ids = list(deputies_info.keys())[:4]
    for deputy_id in sample_deputy_ids:
        print(f"  ID {deputy_id}: {deputies_info[deputy_id]}")
elif isinstance(deputies_info, list):
    for rec in deputies_info[:4]:
        print(f"  ID {rec.get('id')}: {rec}")
else:
    print(f"  Unexpected deputies_info type: {type(deputies_info)}")

Loaded 19134 enriched resolutions
Loaded 46 deputy records
Loaded 24 person-function mappings

Sample summary resolution:
  Keys: dict_keys(['file', 'date', 'volgnr', 'resolution_index', 'text', 'institutions', 'persons', 'places', 'ships', 'secret', 'president_ids', 'deputy_ids'])
  Date: None
  Deputy IDs: []

Sample deputy info:
  ID 494421: {'id': 494421, 'name': 'Rantwyck (Gelderland)', 'province': 'Gelderland'}
  ID 216739: {'id': 216739, 'name': 'De Bije (Gelderland)', 'province': 'Gelderland'}
  ID 635895: {'id': 635895, 'name': 'Bas (Holland)', 'province': 'Holland'}
  ID 716295: {'id': 716295, 'name': 'Bleyswyck (Holland)', 'province': 'Holland'}


In [6]:
# Build lookup tables
# deputy_id -> canonical name
deputy_id_to_name = {}
for rec in deputies_info.values() if isinstance(deputies_info, dict) else deputies_info:
    if isinstance(rec, dict) and 'id' in rec and 'full_name' in rec:
        deputy_id_to_name[rec['id']] = rec['full_name']

# If deputies_info is a list of records
if isinstance(deputies_info, list):
    deputy_id_to_name = {}
    for rec in deputies_info:
        if 'id' in rec and 'full_name' in rec:
            deputy_id_to_name[rec['id']] = rec['full_name']

print(f"Built deputy name lookup: {len(deputy_id_to_name)} entries")

# person_id -> role (from persoon_functie)
person_id_to_role = persoon_functie if isinstance(persoon_functie, dict) else {}
print(f"Person-function lookup: {len(person_id_to_role)} entries")
print(f"Sample roles: {list(person_id_to_role.values())[:3]}")

# Index summaries by date for fast lookup
from collections import defaultdict
summaries_by_date = defaultdict(list)
for summary in summaries:
    summaries_by_date[summary['date']].append(summary)
print(f"Indexed summaries by {len(summaries_by_date)} unique dates")


Built deputy name lookup: 0 entries
Person-function lookup: 24 entries
Sample roles: [{'0': 208391, '1': 394449, '2': 394449, '3': 793632, '4': 494421, '5': 494421, '6': 585614, '7': 585614, '8': 640777, '9': 819382, '10': 411666, '11': 418687, '12': 404298, '13': 208430, '14': 262317, '15': 358391, '16': 472952, '17': 706741, '18': 62104, '19': 846576, '20': 861791, '21': 685986, '22': 580636, '23': 952106, '24': 952106, '25': 510922, '26': 148727, '27': 789176, '28': 789176, '29': 714625, '30': 70899, '31': 840780, '32': 282821, '33': 315850, '34': 154072, '35': 442342, '36': 940514, '37': 90194, '38': 810300, '39': 1007, '40': 457922, '41': 189129, '42': 789512, '43': 789512, '44': 358538, '45': 657448, '46': 657448, '47': 360496, '48': 565711, '49': 951559, '50': 842758, '51': 51447, '52': 168184, '53': 909612, '54': 864148, '55': 727016, '56': 184766, '57': 184766, '58': 811908, '59': 227144, '60': 262711, '61': 415669, '62': 415669, '63': 902003, '64': 93222, '65': 551879, '66': 

In [7]:
summaries_df['date']

NameError: name 'summaries_df' is not defined

In [ ]:
# Filter resolutions_flat to 1626-1630 and prepare for alignment

def to_daily_period(value):
    """Convert historical dates to daily Period values without Timestamp bounds issues."""
    if pd.isna(value):
        return pd.NaT
    try:
        return pd.Period(str(value)[:10], freq='D')
    except (TypeError, ValueError):
        return pd.NaT

resolutions_df['date_period'] = pd.Series(
    [to_daily_period(value) for value in resolutions_df['date']],
    index=resolutions_df.index,
    dtype='period[D]',
)
res_1626_1630 = resolutions_df[
    resolutions_df['date_period'].notna()
    & resolutions_df['date_period'].dt.year.between(1626, 1630)
].copy()
print(f"Filtered to {len(res_1626_1630)} resolutions in 1626-1630")

# Convert summary dates for matching using Periods instead of Timestamps
summaries_df = pd.DataFrame(summaries)
summaries_df['date_period'] = pd.Series(
    [to_daily_period(value) for value in summaries_df['date']],
    index=summaries_df.index,
    dtype='period[D]',
)
summaries_df = summaries_df[summaries_df['date_period'].notna()].copy()
summaries_1626_1630 = summaries_df[
    summaries_df['date_period'].dt.year.between(1626, 1630)
].copy()
print(f"Summaries in 1626-1630: {len(summaries_1626_1630)}")

# Match by exact date first
def find_summary_for_resolution(res_date_period):
    """Find summary matching resolution date, with 1-day tolerance."""
    if pd.isna(res_date_period):
        return None

    matches = summaries_1626_1630[
        summaries_1626_1630['date_period'] == res_date_period
    ].to_dict('records')
    if matches:
        return matches[0]

    # Try ±1 day
    for offset in (-1, 1):
        alt_date = res_date_period + offset
        matches = summaries_1626_1630[
            summaries_1626_1630['date_period'] == alt_date
        ].to_dict('records')
        if matches:
            return matches[0]
    return None

# Add summary link
res_1626_1630['summary'] = res_1626_1630['date_period'].apply(find_summary_for_resolution)
linked = res_1626_1630[res_1626_1630['summary'].notna()]
print(f"Linked {len(linked)} / {len(res_1626_1630)} resolutions to summaries ({100*len(linked)/len(res_1626_1630):.1f}%)")

Filtered to 12811 resolutions in 1626-1630
Summaries in 1626-1630: 19133
Linked 11403 / 12811 resolutions to summaries (89.0%)


In [ ]:
test_res

id                                   session-3186-num-10-resolution-1
type                                                       resolution
date                                                       1627-01-13
year                                                             1627
weekday                                                      woensdag
paragraph_texts     ["Opt versoeck van Claes Cornelissen Meester t...
resolutions_text    Opt versoeck van Claes Cornelissen Meester tim...
date_period                                                1627-01-13
summary             {'file': '162713ja.xml', 'date': '1627-01-13',...
Name: 84104, dtype: object

In [ ]:
def extract_name_spans(text, canonical_names, threshold=80):
    """
    Search for canonical delegate names in HTR text using fuzzy matching.
    Return list of (name, span_text, char_offset, fuzzy_score).
    """
    if not text or not canonical_names:
        return []
    
    spans = []
    # Tokenize text into words for fuzzy matching
    words = text.split()
    
    for canonical in canonical_names:
        # Try exact substring match first (fastest)
        idx = text.lower().find(canonical.lower())
        if idx >= 0:
            spans.append({
                'canonical': canonical,
                'span': canonical,
                'offset': idx,
                'score': 100,
                'method': 'exact'
            })
            continue
        
        # Try fuzzy match on consecutive word n-grams
        canonical_tokens = canonical.split()
        for i in range(len(words) - len(canonical_tokens) + 1):
            ngram = ' '.join(words[i:i+len(canonical_tokens)])
            score = token_set_ratio(canonical, ngram)
            if score >= threshold:
                offset = sum(len(w) + 1 for w in words[:i])  # Approximate
                spans.append({
                    'canonical': canonical,
                    'span': ngram,
                    'offset': offset,
                    'score': score,
                    'method': 'fuzzy'
                })
    
    return spans

# Test on first 20 linked resolution

for i in range(min(20, len(linked))):
    test_res = linked.iloc[i]
    test_summary = test_res['summary']
    test_names = [deputy_id_to_name.get(did) for did in test_summary.get('deputy_ids', []) if did in deputy_id_to_name]
    test_names = [n for n in test_names if n]  # Filter Nones

    if test_res is not None and test_res['paragraph_texts']:
        first_para = test_res['paragraph_texts'][0] if isinstance(test_res['paragraph_texts'], list) else test_res['paragraph_texts']
        test_spans = extract_name_spans(str(first_para), test_names, threshold=75)
        print(f"Test: Found {len(test_spans)} spans for {len(test_names)} canonical names")
        if test_spans:
            print(f"  Examples: {test_spans[:2]}")


Test: Found 0 spans for 0 canonical names
Test: Found 0 spans for 0 canonical names
Test: Found 0 spans for 0 canonical names
Test: Found 0 spans for 0 canonical names
Test: Found 0 spans for 0 canonical names
Test: Found 0 spans for 0 canonical names
Test: Found 0 spans for 0 canonical names
Test: Found 0 spans for 0 canonical names
Test: Found 0 spans for 0 canonical names
Test: Found 0 spans for 0 canonical names
Test: Found 0 spans for 0 canonical names
Test: Found 0 spans for 0 canonical names
Test: Found 0 spans for 0 canonical names
Test: Found 0 spans for 0 canonical names
Test: Found 0 spans for 0 canonical names
Test: Found 0 spans for 0 canonical names
Test: Found 0 spans for 0 canonical names
Test: Found 0 spans for 0 canonical names
Test: Found 0 spans for 0 canonical names
Test: Found 0 spans for 0 canonical names


In [ ]:
# as a first test it may be an idea to try and link on the placenames 
# with the same fuzzy matching approach, to see if we can get a better match rate 
# on the summaries.

# so we can try and extract the placenames from the summaries, and then see if we can find those in the resolution text, to link them together.



In [ ]:
summaries_df

,file,date,volgnr,resolution_index,text,institutions,persons,places,ships,secret,president_ids,deputy_ids,date_period
1,163003ap.xml,1630-04-03,None,0,Het rapport van Huijgens en andere gedeputeerd...,[],[791967],[Emden],[],False,[494421],"[494421, 216739, 208391, 296035, 791967, 27788...",1630-04-03
2,163003ap.xml,1630-04-03,None,1,De RvS zal de provincies aanschrijven de kapit...,[],[],[],[],False,[494421],"[494421, 216739, 208391, 296035, 791967, 27788...",1630-04-03
3,163003ap.xml,1630-04-03,None,2,Schaffer rapporteert conform de resolutie van ...,[],"[168184, 124646, 125152]",[],[],False,[494421],"[494421, 216739, 208391, 296035, 791967, 27788...",1630-04-03
4,163003ap.xml,1630-04-03,None,3,President Rantwijck deelt mee dat ambassadeur ...,[],"[494421, 930606, 727188]",[],[],False,[494421],"[494421, 216739, 208391, 296035, 791967, 27788...",1630-04-03
5,163003ap.xml,1630-04-03,None,4,HHM lezen de deductie van Govert Govertsz. Boo...,[],"[920048, 729000, 614058]",[],[],False,[494421],"[494421, 216739, 208391, 296035, 791967, 27788...",1630-04-03
...,...,...,...,...,...,...,...,...,...,...,...,...,...
19129,162607juli.xml,1626-07-07,None,11,"Dirck Abbas en Joris Sforcen, generaals\n van ...",[],"[404298, 93222]",[Calais],[],False,[418011],"[387460, 360496, 938346, 962163, 392307, 58561...",1626-07-07
19130,162607juli.xml,1626-07-07,None,12,Orateur Haga schrijft d.d. Constantinopel [Ist...,[],"[640580, 1025]",[],[],False,[418011],"[387460, 360496, 938346, 962163, 392307, 58561...",1626-07-07
19131,162607juli.xml,1626-07-07,None,13,De gedeputeerden van Holland hebben opnieuw aa...,[],[],[Holland],[],False,[418011],"[387460, 360496, 938346, 962163, 392307, 58561...",1626-07-07
19132,162607juli.xml,1626-07-07,None,14,De gedeputeerden van Holland hebben aangedrong...,[],"[386973, 387460, 360496, 490592, 938346, 418011]",[Holland],[],False,[418011],"[387460, 360496, 938346, 962163, 392307, 58561...",1626-07-07


In [ ]:
# this does not work well, probably due to date mismatches and HTR noise, 
# does the fuzzy matching find any of the canonical names in the resolution text? 
# Lets  check

for i in range(min(20, len(linked))):
    test_res = linked.iloc[i]
    test_summary = test_res['summary']
    test_names = [deputy_id_to_name.get(did) for did in test_summary.get('deputy_ids', []) if did in deputy_id_to_name]
    test_names = [n for n in test_names if n]  # Filter Nones

    if test_res is not None and test_res['paragraph_texts']:
        first_para = test_res['paragraph_texts'][0] if isinstance(test_res['paragraph_texts'], list) else test_res['paragraph_texts']
        test_spans = extract_name_spans(str(first_para), test_names, threshold=75)
        print(f"Test: Found {len(test_spans)} spans for {len(test_names)} canonical names")
        if test_spans:
            print(f"  Examples: {test_spans[:2]}")



Test: Found 0 spans for 0 canonical names
Test: Found 0 spans for 0 canonical names
Test: Found 0 spans for 0 canonical names
Test: Found 0 spans for 0 canonical names
Test: Found 0 spans for 0 canonical names
Test: Found 0 spans for 0 canonical names
Test: Found 0 spans for 0 canonical names
Test: Found 0 spans for 0 canonical names
Test: Found 0 spans for 0 canonical names
Test: Found 0 spans for 0 canonical names
Test: Found 0 spans for 0 canonical names
Test: Found 0 spans for 0 canonical names
Test: Found 0 spans for 0 canonical names
Test: Found 0 spans for 0 canonical names
Test: Found 0 spans for 0 canonical names
Test: Found 0 spans for 0 canonical names
Test: Found 0 spans for 0 canonical names
Test: Found 0 spans for 0 canonical names
Test: Found 0 spans for 0 canonical names
Test: Found 0 spans for 0 canonical names


In [ ]:
# Place-based linking as alternative to date matching
# Extract place names from summaries and search for them in resolution text

def extract_places_from_summary(summary):
    """Extract place names from summary."""
    places = summary.get('places', [])
    if isinstance(places, list):
        return [p.lower() if isinstance(p, str) else str(p).lower() for p in places if p]
    return []

def find_place_overlap(res_text, summary_places, threshold=75):
    """Find place name matches in resolution text using fuzzy matching."""
    if not res_text or not summary_places:
        return []
    
    text_lower = str(res_text).lower()
    matches = []
    
    for place in summary_places:
        # Exact substring match
        if place in text_lower:
            matches.append({'place': place, 'method': 'exact', 'score': 100})
        else:
            # Fuzzy match on tokens
            place_tokens = place.split()
            text_words = text_lower.split()
            for i in range(len(text_words) - len(place_tokens) + 1):
                ngram = ' '.join(text_words[i:i+len(place_tokens)])
                score = token_set_ratio(place, ngram)
                if score >= threshold:
                    matches.append({'place': place, 'ngram': ngram, 'method': 'fuzzy', 'score': score})
    
    return matches

# Try place-based linking for resolutions not linked by date
print("=== Place-Based Linking ===")

# Extract places from all summaries
summaries_with_places = []
for summary in summaries:
    places = extract_places_from_summary(summary)
    if places:
        summaries_with_places.append({
            'date': summary.get('date'),
            'places': places,
            'summary': summary
        })

print(f"Summaries with places: {len(summaries_with_places)} / {len(summaries)}")

# For resolutions not yet linked, try place matching
unlinked = res_1626_1630[res_1626_1630['summary'].isna()].copy()
print(f"Unlinked resolutions to try: {len(unlinked)}")

place_linked = []
for idx, (res_idx, res) in enumerate(unlinked.iterrows()):
    if idx % 1000 == 0:
        print(f"  Checking {idx}/{len(unlinked)}...")
    
    # Concatenate all paragraphs for search
    text = ' '.join([str(p) for p in res['paragraphs_texts']]) if res['paragraphs_texts'] else ''
    
    # Try each summary with places
    best_match = None
    best_score = 0
    
    for summary_rec in summaries_with_places:
        matches = find_place_overlap(text, summary_rec['places'], threshold=80)
        if matches:
            avg_score = sum(m['score'] for m in matches) / len(matches)
            if avg_score > best_score:
                best_score = avg_score
                best_match = summary_rec['summary']
    
    if best_match and best_score >= 85:  # High threshold for place matching
        place_linked.append({
            'resolution_idx': res_idx,
            'summary': best_match,
            'place_score': best_score
        })

print(f"Place-based linked: {len(place_linked)} additional resolutions")

# Merge place-linked back
for link in place_linked:
    unlinked.loc[link['resolution_idx'], 'summary'] = link['summary']

# Recount
linked_updated = res_1626_1630[res_1626_1630['summary'].notna()]
print(f"\n=== Linking Summary ===")
print(f"Total linked (date + place): {len(linked_updated)} / {len(res_1626_1630)} ({100*len(linked_updated)/len(res_1626_1630):.1f}%)")
print(f"  Date-based: {len(linked)}")
print(f"  Place-based addition: {len(place_linked)}")

linked = linked_updated
    'resolutions_processed': 0,
    'names_looked_up': 0,
    'spans_found': 0,
    'high_confidence': 0,  # score >= 90
    'fuzzy_matches': 0,
}

for idx, (res_idx, res) in enumerate(linked.iterrows()):
    if idx % 1000 == 0:
        print(f"Processing {idx}/{len(linked)}...")
    
    summary = res['summary']
    deputy_ids = summary.get('deputy_ids', [])
    
    # Get canonical names
    canonical_names = []
    for did in deputy_ids:
        if did in deputy_id_to_name:
            canonical_names.append(deputy_id_to_name[did])
    
    if not canonical_names:
        continue
    
    stats['names_looked_up'] += len(canonical_names)
    
    # Search paragraphs
    paragraphs = res['paragraphs_texts']
    if not paragraphs:
        continue
    
    if not isinstance(paragraphs, list):
        paragraphs = [paragraphs]
    
    for para_idx, para in enumerate(paragraphs):
        if not para:
            continue
        
        spans = extract_name_spans(str(para), canonical_names, threshold=75)
        
        for span_info in spans:
            canonical = span_info['canonical']
            score = span_info['score']
            role = person_id_to_role.get(canonical, "unknown")  # Try to get role
            
            training_pairs.append({
                'htr_span': span_info['span'],
                'canonical_name': canonical,
                'role': role,
                'score': score,
                'method': span_info['method'],
                'date': res['date'],
                'resolution_id': res.get('volgnr'),
                'paragraph_idx': para_idx,
            })
            
            stats['spans_found'] += 1
            if score >= 90:
                stats['high_confidence'] += 1
            if span_info['method'] == 'fuzzy':
                stats['fuzzy_matches'] += 1
    
    stats['resolutions_processed'] += 1

print("\n=== Extraction complete ===")
print(f"Statistics: {stats}")
print(f"\nTraining pairs generated: {len(training_pairs)}")
if training_pairs:
    print(f"Sample pairs:")
    for pair in training_pairs[:3]:
        print(f"  HTR: '{pair['htr_span']}' -> {pair['canonical_name']} (role: {pair['role']}, score: {pair['score']})")


Processing 0/11403...
Processing 1000/11403...
Processing 2000/11403...
Processing 3000/11403...
Processing 4000/11403...
Processing 5000/11403...
Processing 6000/11403...
Processing 7000/11403...
Processing 8000/11403...
Processing 9000/11403...
Processing 10000/11403...
Processing 11000/11403...

=== Extraction complete ===
Statistics: {'resolutions_processed': 0, 'names_looked_up': 0, 'spans_found': 0, 'high_confidence': 0, 'fuzzy_matches': 0}

Training pairs generated: 0


In [ ]:
import json

# Save training pairs
training_df = pd.DataFrame(training_pairs)

output_parquet = datadir / "training_pairs_1626_1630.parquet"
training_df.to_parquet(output_parquet)
print(f"Saved {len(training_df)} pairs to {output_parquet}")

# Save as JSON-L for easy inspection
output_jsonl = datadir / "training_pairs_1626_1630.jsonl"
with open(output_jsonl, 'w') as f:
    for _, row in training_df.iterrows():
        f.write(json.dumps(row.to_dict(), default=str) + '\n')
print(f"Also saved as JSON-L: {output_jsonl}")

# Quality analysis
print("\n=== Quality Analysis ===")
print(f"Total pairs: {len(training_df)}")
print(f"Unique canonical names: {training_df['canonical_name'].nunique()}")
print(f"Unique HTR variants: {training_df['htr_span'].nunique()}")
print(f"\nScore distribution:")
print(f"  Exact (100): {(training_df['score'] == 100).sum()}")
print(f"  High (90-100): {((training_df['score'] >= 90) & (training_df['score'] < 100)).sum()}")
print(f"  Medium (80-90): {((training_df['score'] >= 80) & (training_df['score'] < 90)).sum()}")
print(f"  Low (75-80): {((training_df['score'] >= 75) & (training_df['score'] < 80)).sum()}")

print(f"\nMethod breakdown:")
print(training_df['method'].value_counts())

print(f"\nTop 10 canonical names by frequency:")
print(training_df['canonical_name'].value_counts().head(10))

print(f"\nTop HTR variants (example noise patterns):")
print(training_df['htr_span'].value_counts().head(5))


Saved 0 pairs to data/training_pairs_1626_1630.parquet
Also saved as JSON-L: data/training_pairs_1626_1630.jsonl

=== Quality Analysis ===
Total pairs: 0


KeyError: 'canonical_name'

In [8]:
# Manual validation: spot-check low-confidence fuzzy matches
print("=== Manual Review: Low Confidence Fuzzy Matches ===")
fuzzy_low = training_df[(training_df['method'] == 'fuzzy') & (training_df['score'] < 85)].sort_values('score')
print(f"Found {len(fuzzy_low)} fuzzy matches with score < 85")

if len(fuzzy_low) > 0:
    print(f"\nSample (first 10):")
    for idx, row in fuzzy_low.head(10).iterrows():
        print(f"  {row['htr_span']:30} -> {row['canonical_name']:20} (score: {row['score']:.0f}%)")

# Deduplication check
print("\n=== Deduplication ===")
print(f"Rows before dedup: {len(training_df)}")
# Keep highest score per (canonical_name, htr_span, date) combination
dedup_df = training_df.sort_values('score', ascending=False).drop_duplicates(
    subset=['canonical_name', 'htr_span', 'date'], 
    keep='first'
)
print(f"Rows after dedup: {len(dedup_df)}")
print(f"Removed {len(training_df) - len(dedup_df)} duplicates")

# Save deduplicated
output_dedup = datadir / "training_pairs_1626_1630_dedup.parquet"
dedup_df.to_parquet(output_dedup)
print(f"Saved deduplicated set to {output_dedup}")


=== Manual Review: Low Confidence Fuzzy Matches ===


NameError: name 'training_df' is not defined

In [9]:
summaries_df.columns

NameError: name 'summaries_df' is not defined

In [ ]:
## Standalone Script

The pipeline logic has been extracted to `build_training_pairs.py` for reproducible execution and integration into workflows.

### Run the script:
```bash
uv run python build_training_pairs.py
```

### Output files:
- `data/training_pairs_1626_1630.parquet` — raw pairs with all variants
- `data/training_pairs_1626_1630.jsonl` — same, line-delimited for inspection
- `data/training_pairs_1626_1630_dedup.parquet` — deduplicated (highest score per canonical + span + date)


In [ ]:
# Load and inspect the generated training pairs
import subprocess
import sys

# Run the script if not already run
script_path = Path("build_training_pairs.py")
if script_path.exists():
    print("Running build_training_pairs.py...\n")
    result = subprocess.run(
        [sys.executable, str(script_path)],
        cwd=Path.cwd(),
        capture_output=False,
        text=True
    )
    print(f"\nScript completed with exit code {result.returncode}")


In [ ]:
# Load and analyze results
dedup_output = datadir / "training_pairs_1626_1630_dedup.parquet"
if dedup_output.exists():
    results_df = pd.read_parquet(dedup_output)
    print(f"Loaded {len(results_df)} training pairs (deduplicated)")
    print(f"\nTop canonical names:")
    print(results_df['canonical_name'].value_counts().head(10))
    print(f"\nScore distribution:")
    print(f"  Exact (100): {(results_df['score'] == 100).sum()}")
    print(f"  High (90-99): {((results_df['score'] >= 90) & (results_df['score'] < 100)).sum()}")
    print(f"  Medium (80-89): {((results_df['score'] >= 80) & (results_df['score'] < 90)).sum()}")
    print(f"  Low (75-79): {((results_df['score'] >= 75) & (results_df['score'] < 80)).sum()}")
else:
    print("No training pairs file found. Run the script first.")


No training pairs file found. Run the script first.
